# Download PR review comment dataset

Pipeline: GH Archive → filter by top snapshot commits → GitHub compare/zipball enrichment → annotated patched content.

Set `GITHUB_TOKEN` in `.env` (or the environment) before enrichment steps. Install the package editable: `pip install -e .` from the repo root.

In [1]:
import asyncio
import logging

import aiohttp
from dotenv import load_dotenv

from ai_code_reviewer.dataset import (
    checkpoints,
    gh_archive,
    github_api,
    patches,
)
from ai_code_reviewer.dataset import config as dataset_config

load_dotenv()
logging.basicConfig(level=logging.INFO)

In [2]:
gh_archive_semaphore = asyncio.Semaphore(dataset_config.GH_ARCHIVE_CONCURRENCY)
gh_semaphore = asyncio.Semaphore(dataset_config.GITHUB_API_CONCURRENCY)

In [3]:
async with aiohttp.ClientSession(
    timeout=dataset_config.default_client_timeout()
) as session:
    dataset = await gh_archive.fetch_pr_comments_range(
        session,
        dataset_config.RANGE_START,
        dataset_config.RANGE_END,
        gh_archive_semaphore,
    )

checkpoints.save_dataset_checkpoint(dataset, dataset_config.DATASET_RAW_PATH)

Fetching hourly data: 100%|██████████| 1/1 [00:23<00:00, 23.62s/it]
INFO:ai_code_reviewer.dataset.checkpoints:Wrote checkpoint -> /Users/nikita/University/DataMining/Project/ai-code-reviewer/data/checkpoints/dataset_raw.json.gz


PosixPath('/Users/nikita/University/DataMining/Project/ai-code-reviewer/data/checkpoints/dataset_raw.json.gz')

In [4]:
dataset = gh_archive.filter_dataset_by_top_snapshot_commits(
    dataset, dataset_config.SNAPSHOT_COMMITS_TO_KEEP
)

checkpoints.save_dataset_checkpoint(dataset, dataset_config.DATASET_FILTERED_PATH)

INFO:ai_code_reviewer.dataset.checkpoints:Wrote checkpoint -> /Users/nikita/University/DataMining/Project/ai-code-reviewer/data/checkpoints/filtered.json.gz


PosixPath('/Users/nikita/University/DataMining/Project/ai-code-reviewer/data/checkpoints/filtered.json.gz')

In [5]:
async with aiohttp.ClientSession(
    timeout=dataset_config.default_client_timeout()
) as session:
    await github_api.enrich_dataset_with_base_and_patches(
        dataset, session, gh_semaphore
    )

checkpoints.save_dataset_checkpoint(dataset, dataset_config.DATASET_ENRICHED_PATH)

100%|██████████| 1/1 [00:03<00:00,  3.35s/it]
INFO:ai_code_reviewer.dataset.checkpoints:Wrote checkpoint -> /Users/nikita/University/DataMining/Project/ai-code-reviewer/data/checkpoints/enriched.json.gz


PosixPath('/Users/nikita/University/DataMining/Project/ai-code-reviewer/data/checkpoints/enriched.json.gz')

In [6]:
patches.enrich_dataset_with_patched_content(dataset)

checkpoints.save_dataset_checkpoint(dataset, dataset_config.DATASET_FINAL_PATH)

INFO:ai_code_reviewer.dataset.checkpoints:Wrote checkpoint -> /Users/nikita/University/DataMining/Project/ai-code-reviewer/data/checkpoints/final.json.gz


PosixPath('/Users/nikita/University/DataMining/Project/ai-code-reviewer/data/checkpoints/final.json.gz')